In [4]:
# -*- coding: utf-8 -*-
"""
Ultimate Final Edition: Hardcoded Best Hyperparameters
Features: 
1. 100% Bulletproof: Uses hardcoded best parameters (No external CSV needed).
2. Generates JAMA-style Figure 3 and Word Table.
3. Trains and exports dual production models (Stacking & XGBoost) for Web App.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast
import warnings
import joblib 
from docx import Document

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, precision_score, f1_score, confusion_matrix, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import BaseEstimator, ClassifierMixin, clone

# --- Plotting Style ---
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 300
warnings.filterwarnings('ignore')

# --- Configuration ---
BASE_DIR = os.getcwd()
TRAIN_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'train_imputed_random_forest.csv')
RFE_LISTS_FOLDER = os.path.join(BASE_DIR, 'RFE_Final_Run', 'feature_lists')
OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized')

TARGET_COL = 'PostopAKI'
SEED = 42

COLORS = {
    'LR': '#D55E00', 'DT': '#0072B2', 'RF': '#009E73', 
    'SVM': '#CC79A7', 'KNN': '#F0E442', 'NB': '#56B4E9', 
    'XGB': '#E69F00', 'SGBT': '#999999', 'NNET': '#000000',
    'Voting': '#882255', 'Stacking': '#117733'
}

# =======================================================================
# 🌟 核心升级：将你找到的最优参数直接硬编码，彻底告别文件读取报错！ 🌟
# =======================================================================
EXACT_BEST_PARAMS = {
    'LR': {'C': 0.03228816990646484},
    'DT': {'max_depth': 5, 'min_samples_split': 8},
    'RF': {'max_depth': None, 'n_estimators': 472},
    'KNN': {'n_neighbors': 25},
    'SVM': {'C': 0.35259619332467856, 'gamma': 0.002437769858811227},
    'NB': {'var_smoothing': 0.09997626858641415},
    'XGB': {'learning_rate': 0.017234847896145766, 'max_depth': 9, 'n_estimators': 220},
    'SGBT': {'learning_rate': 0.017897945928910076, 'max_depth': 3, 'n_estimators': 100},
    'NNET': {'alpha': 0.004782115691629105, 'hidden_layer_sizes': (100,)} # 已将 str 转换为 tuple
}

# --- Helper Classes ---
def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    return thresholds[np.argmax(tpr - fpr)]

def save_word_table(df, filename, title):
    doc = Document()
    doc.add_heading(title, level=1)
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    hdr_cells = table.rows[0].cells
    for i, col in enumerate(df.columns): 
        hdr_cells[i].text = str(col)
        for p in hdr_cells[i].paragraphs: 
            for r in p.runs: r.font.bold = True
    for _, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, val in enumerate(row):
            row_cells[i].text = str(val)
    doc.save(os.path.join(OUTPUT_DIR, filename))

# --- Main Logic ---
def main():
    if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

    print("=== 1. Loading Training Data ===")
    try: df_train = pd.read_csv(TRAIN_FILE, encoding='gbk')
    except: 
        try: df_train = pd.read_csv(TRAIN_FILE, encoding='utf-8')
        except Exception as e:
            print(f"❌ 找不到训练数据 {TRAIN_FILE}，请检查路径。")
            return
    
    if 'Unnamed: 0' in df_train.columns: df_train.drop(columns=['Unnamed: 0'], inplace=True)
    if 'ID' in df_train.columns: df_train.drop(columns=['ID'], inplace=True) 
    
    y = df_train[TARGET_COL].astype(int)
    exclude = [TARGET_COL, 'Patient_ID', 'Center', 'AKIStage']
    X = df_train.drop(columns=[c for c in exclude if c in df_train.columns])
    X = pd.get_dummies(X, drop_first=True)
    
    print(f"  Training Set: {X.shape}")

    print("\n=== 2. Setting up Models with EXACT Best Hyperparameters ===")
    models_map = {
        'LR': LogisticRegression(solver='liblinear', random_state=SEED),
        'DT': DecisionTreeClassifier(random_state=SEED),
        'RF': RandomForestClassifier(random_state=SEED),
        'KNN': KNeighborsClassifier(),
        'SVM': SVC(probability=True, random_state=SEED),
        'NB': GaussianNB(),
        'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=SEED),
        'SGBT': GradientBoostingClassifier(random_state=SEED),
        'NNET': MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED) # 直接使用原生 MLP
    }

    final_pipelines = {}
    
    for name, model in models_map.items():
        feat_file = os.path.join(RFE_LISTS_FOLDER, f'selected_features_{name}_Combined.txt')
        valid_feats = X.columns.tolist()
        if os.path.exists(feat_file):
            with open(feat_file, 'r') as f: feats = [l.strip() for l in f if l.strip()]
            temp = [f for f in feats if f in X.columns]
            if temp: valid_feats = temp

        # 🌟 直接赋予最优参数 🌟
        if name in EXACT_BEST_PARAMS:
            model.set_params(**EXACT_BEST_PARAMS[name])

        pipeline = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=SEED)), ('classifier', model)])
        final_pipelines[name] = (pipeline, valid_feats)

    print("\n=== 3. Running 10-fold Cross-Validation ===")
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_metrics = {'Model': [], 'AUC': [], 'Spec': [], 'Sens': []}
    
    fold = 1
    for train_idx, val_idx in skf.split(X, y):
        print(f"  Processing Fold {fold}/10...", end='\r')
        
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        fold_models = {}
        fold_preds_val = {}
        fold_oof_train_preds = pd.DataFrame(index=y_tr.index)
        
        for name, (pipe, feats) in final_pipelines.items():
            X_tr, X_val = X.iloc[train_idx][feats], X.iloc[val_idx][feats]
            clf = clone(pipe)
            clf.fit(X_tr, y_tr)
            
            prob_val = clf.predict_proba(X_val)[:, 1]
            fold_preds_val[name] = prob_val
            
            auc_v = roc_auc_score(y_val, prob_val)
            th_v = find_optimal_threshold(y_val, prob_val)
            y_p_v = (prob_val >= th_v).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_val, y_p_v).ravel()
            
            cv_metrics['Model'].append(name)
            cv_metrics['AUC'].append(auc_v)
            cv_metrics['Spec'].append(tn/(tn+fp))
            cv_metrics['Sens'].append(tp/(tp+fn))
            
            clf_oof = clone(pipe)
            oof_tr = cross_val_predict(clf_oof, X_tr, y_tr, cv=3, method='predict_proba', n_jobs=-1)[:, 1]
            fold_oof_train_preds[name] = oof_tr
        
        fold_aucs = {n: roc_auc_score(y_val, fold_preds_val[n]) for n in models_map}
        top3 = sorted(fold_aucs, key=fold_aucs.get, reverse=True)[:3]
        top4 = sorted(fold_aucs, key=fold_aucs.get, reverse=True)[:4]
        
        vote_prob = np.mean([fold_preds_val[n] for n in top3], axis=0)
        th_vote = find_optimal_threshold(y_val, vote_prob)
        y_p_vote = (vote_prob >= th_vote).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_p_vote).ravel()
        
        cv_metrics['Model'].append('Voting')
        cv_metrics['AUC'].append(roc_auc_score(y_val, vote_prob))
        cv_metrics['Spec'].append(tn/(tn+fp))
        cv_metrics['Sens'].append(tp/(tp+fn))
        
        meta_X_tr = fold_oof_train_preds[top4]
        meta_X_val = pd.DataFrame({n: fold_preds_val[n] for n in top4})
        
        meta_learner = LogisticRegressionCV(cv=3, random_state=SEED, scoring='roc_auc')
        meta_learner.fit(meta_X_tr, y_tr)
        
        stack_prob = meta_learner.predict_proba(meta_X_val)[:, 1]
        th_stack = find_optimal_threshold(y_val, stack_prob)
        y_p_stack = (stack_prob >= th_stack).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, y_p_stack).ravel()
        
        cv_metrics['Model'].append('Stacking')
        cv_metrics['AUC'].append(roc_auc_score(y_val, stack_prob))
        cv_metrics['Spec'].append(tn/(tn+fp))
        cv_metrics['Sens'].append(tp/(tp+fn))
        
        fold += 1

    print("\n  Processing complete.")

    print("\n=== 4. Generating Figure 3 ===")
    df_plot = pd.DataFrame(cv_metrics)
    order = df_plot.groupby('Model')['AUC'].mean().sort_values(ascending=False).index
    df_plot['Model'] = pd.Categorical(df_plot['Model'], categories=order, ordered=True)
    df_plot = df_plot.sort_values('Model')
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 10), sharey=False)
    
    def draw_panel(ax, metric_col, title):
        sns.boxplot(data=df_plot, y='Model', x=metric_col, ax=ax, palette=COLORS, 
                    orient='h', showfliers=False, width=0.5, linewidth=1)
        sns.stripplot(data=df_plot, y='Model', x=metric_col, ax=ax, color='black', 
                      alpha=0.3, size=3, jitter=True)
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_xlim(0, 1.1)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        
        stats = df_plot.groupby('Model')[metric_col].agg(['mean', 'std', 'sem'])
        stats['ci95_lo'] = stats['mean'] - 1.96 * stats['sem']
        stats['ci95_hi'] = stats['mean'] + 1.96 * stats['sem']
        
        for i, model in enumerate(order):
            row = stats.loc[model]
            text_mean = f"{row['mean']:.3f}±{row['std']:.3f}"
            text_ci = f"95% CI({row['ci95_lo']:.3f}-{row['ci95_hi']:.3f})"
            ax.text(0.05, i - 0.15, text_mean, fontsize=10, fontweight='bold', va='center')
            ax.text(0.05, i + 0.15, text_ci, fontsize=9, color='gray', va='center')

    draw_panel(axes[0], 'AUC', 'ROC')
    draw_panel(axes[1], 'Spec', 'Specificity')
    draw_panel(axes[2], 'Sens', 'Sensitivity')
    
    plt.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'Figure3_Internal_Validation.png'), dpi=300)
    plt.close()
    
    summary = df_plot.groupby('Model').agg(['mean', 'std']).reset_index()
    summary.columns = ['Model'] + [f"{c[0]}_{c[1]}" for c in summary.columns[1:]]
    
    final_table = pd.DataFrame()
    final_table['Model'] = summary['Model']
    final_table['AUC'] = summary.apply(lambda r: f"{r['AUC_mean']:.3f} ± {r['AUC_std']:.3f}", axis=1)
    final_table['Specificity'] = summary.apply(lambda r: f"{r['Spec_mean']:.3f} ± {r['Spec_std']:.3f}", axis=1)
    final_table['Sensitivity'] = summary.apply(lambda r: f"{r['Sens_mean']:.3f} ± {r['Sens_std']:.3f}", axis=1)
    
    save_word_table(final_table, 'Table_Internal_Validation.docx', 'Internal Validation Results (10-fold CV)')

    # =====================================================================
    # === 6. Training & Saving Final Production Models (For Web App) ===
    # =====================================================================
    print("\n=== 6. Training & Saving Final Production Models ===")
    
    web_features = ['Age', 'Gender', 'BMI', 'Diabetes', 'PreopAlb', 'OperationTime', 
                    'NonOpTransfusion', 'IntraopPlasma', 'NonOpAlbumin', 'PerioperativeVasoactive', 
                    'TNM_Stage', 'Gastrocolorectal']
    
    available_cols = []
    for f in web_features:
        matched = [c for c in df_train.columns if f.lower() in c.lower()]
        if matched: available_cols.append(matched[0])
    
    if len(available_cols) > 0:
        X_web = df_train[available_cols].fillna(0)
        
        print("Training final Stacking model for Overall AKI on 100% data...")
        # 直接提取硬编码的最优参数喂给网页实战模型
        base_estimators = [
            ('rf', RandomForestClassifier(**EXACT_BEST_PARAMS['RF'], random_state=SEED)),
            ('xgb', XGBClassifier(**EXACT_BEST_PARAMS['XGB'], use_label_encoder=False, eval_metric='logloss', random_state=SEED)),
            ('lr', LogisticRegression(**EXACT_BEST_PARAMS['LR'], solver='liblinear', random_state=SEED))
        ]
        final_stacking_clf = StackingClassifier(estimators=base_estimators, final_estimator=LogisticRegression(), cv=5)
        final_stacking_clf.fit(X_web, y)
        
        stacking_path = os.path.join(OUTPUT_DIR, 'model_stacking.pkl')
        joblib.dump((final_stacking_clf, available_cols), stacking_path) 
        print(f"✅ Stacking 部署模型已保存至: {stacking_path}")

        if 'AKIStage' in df_train.columns:
            print("Training final XGBoost model for Severe AKI...")
            y_severe = (df_train['AKIStage'] >= 2).astype(int)
            final_xgb_severe = XGBClassifier(**EXACT_BEST_PARAMS['XGB'], use_label_encoder=False, eval_metric='logloss', random_state=SEED)
            final_xgb_severe.fit(X_web, y_severe)
            
            xgb_path = os.path.join(OUTPUT_DIR, 'model_xgboost_severe.pkl')
            joblib.dump((final_xgb_severe, available_cols), xgb_path)
            print(f"✅ XGBoost 重症模型已保存至: {xgb_path}")
        else:
            print("⚠️ 找不到 'AKIStage' 列，跳过重症模型训练。")
    else:
        print("⚠️ 无法在数据集中找到网页所需的12个特征，部署模型生成失败。")

    print("\n🎉 极其完美！全部运行完成！前往文件夹拿 .pkl 文件去启动你的 JAMA 级双引擎 Web App 吧！")

if __name__ == '__main__':
    main()

=== 1. Loading Training Data ===
  Training Set: (2446, 138)

=== 2. Setting up Models with EXACT Best Hyperparameters ===

=== 3. Running 10-fold Cross-Validation ===
  Processing Fold 10/10...
  Processing complete.

=== 4. Generating Figure 3 ===

=== 6. Training & Saving Final Production Models ===
Training final Stacking model for Overall AKI on 100% data...
✅ Stacking 部署模型已保存至: /home/lei/AKI新分析/Final_Internal_Validation_Optimized/model_stacking.pkl
⚠️ 找不到 'AKIStage' 列，跳过重症模型训练。

🎉 极其完美！全部运行完成！前往文件夹拿 .pkl 文件去启动你的 JAMA 级双引擎 Web App 吧！


In [7]:
import pandas as pd
import joblib
from xgboost import XGBClassifier
import os

BASE_DIR = os.getcwd()
TRAIN_FILE = os.path.join(BASE_DIR, 'imputation', 'imputed_data', 'train_imputed_random_forest.csv')
ORIGINAL_FILE = os.path.join(BASE_DIR, 'inter_eng.csv') # 你的原始数据文件
OUTPUT_DIR = os.path.join(BASE_DIR, 'Final_Internal_Validation_Optimized')

print("=== 正在单独补跑 XGBoost 重症模型 ===")

# 1. 🌟 智能加载数据 (解决 UnicodeDecodeError) 🌟
print("正在读取文件...")
try:
    df_train = pd.read_csv(TRAIN_FILE, encoding='gbk')
except UnicodeDecodeError:
    df_train = pd.read_csv(TRAIN_FILE, encoding='utf-8')

try:
    df_orig = pd.read_csv(ORIGINAL_FILE, encoding='gbk')
except UnicodeDecodeError:
    try:
        df_orig = pd.read_csv(ORIGINAL_FILE, encoding='utf-8')
    except UnicodeDecodeError:
        df_orig = pd.read_csv(ORIGINAL_FILE, encoding='latin1') # 终极兜底格式
print("读取成功！")

# 2. 对齐 ID 并提取 AKIStage
if 'Unnamed: 0' in df_train.columns: 
    df_train.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
if 'ID' not in df_orig.columns: 
    df_orig['ID'] = df_orig.index

df_train['ID'] = df_train['ID'].astype(str)
df_orig['ID'] = df_orig['ID'].astype(str)

# 寻找原始数据中的分期列 (可能叫 AKIStage, Stage, AKI_Stage 等)
stage_col = [c for c in df_orig.columns if 'stage' in c.lower() and 'tnm' not in c.lower() and 't_' not in c.lower()]

if len(stage_col) > 0:
    stage_name = stage_col[0]
    print(f"✅ 在原始数据中找到分期列: '{stage_name}'")
    
    # 合并数据
    df_merged = pd.merge(df_train, df_orig[['ID', stage_name]], on='ID', how='left')
    df_merged.dropna(subset=[stage_name], inplace=True) # 清除没有分期记录的行
    
    # 提取网页用的 12 个特征
    web_features = ['Age', 'Gender', 'BMI', 'Diabetes', 'PreopAlb', 'OperationTime', 
                    'NonOpTransfusion', 'IntraopPlasma', 'NonOpAlbumin', 'PerioperativeVasoactive', 
                    'TNM_Stage', 'Gastrocolorectal']
    available_cols = []
    for f in web_features:
        matched = [c for c in df_merged.columns if f.lower() in c.lower()]
        if matched: available_cols.append(matched[0])
        
    X_web = df_merged[available_cols].fillna(0)
    
    # 定义重症：分期 >= 2
    y_severe = (df_merged[stage_name] >= 2).astype(int)
    print(f"📌 提取到 {y_severe.sum()} 例重症患者。开始训练...")
    
    # 使用你找到的最优参数训练
    xgb_params = {'learning_rate': 0.017234847896145766, 'max_depth': 9, 'n_estimators': 220}
    model = XGBClassifier(**xgb_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X_web, y_severe)
    
    # 保存模型
    xgb_path = os.path.join(OUTPUT_DIR, 'model_xgboost_severe.pkl')
    joblib.dump((model, available_cols), xgb_path)
    print(f"🎉 成功！XGBoost 重症模型已补齐，保存在: {xgb_path}")
    print("👉 现在你可以带着两个 .pkl 文件去运行 Web App 网页了！")
else:
    print("❌ 原始数据中没有找到 AKI 分期列，无法训练重症模型。请使用【路径 B】的单引擎网页代码。")

=== 正在单独补跑 XGBoost 重症模型 ===
正在读取文件...
读取成功！
✅ 在原始数据中找到分期列: 'AKIStage'
📌 提取到 19 例重症患者。开始训练...
🎉 成功！XGBoost 重症模型已补齐，保存在: /home/lei/AKI新分析/Final_Internal_Validation_Optimized/model_xgboost_severe.pkl
👉 现在你可以带着两个 .pkl 文件去运行 Web App 网页了！


In [8]:
import streamlit as st
import pandas as pd
import joblib
import os

# ==========================================
# 1. 页面基本设置与顶刊级文案
# ==========================================
st.set_page_config(page_title="Postop AKI Risk Predictor", page_icon="🏥", layout="wide")

st.title("Postoperative AKI & Severe AKI Risk Prediction System")
st.markdown("""
**Disclaimer:** This interactive dual-model system is designed to assist clinicians in predicting the individualized risk of Acute Kidney Injury (AKI). 
- **Primary Screening:** An optimized **Stacking Ensemble** predicts the overall risk of developing AKI.
- **Severity Assessment:** An **XGBoost Algorithm** specifically predicts the risk of developing Severe AKI (Stage II/III).
*Developed in adherence to the TRIPOD-AI reporting guidelines.*
""")
st.divider()

# ==========================================
# 2. 模型加载逻辑 (带安全保护)
# ==========================================
@st.cache_resource
def load_models():
    stacking_path = "model_stacking.pkl"
    xgb_path = "model_xgboost_severe.pkl"
    
    if not os.path.exists(stacking_path) or not os.path.exists(xgb_path):
        return None, None, None
    
    # 我们在训练时把模型和所需的列名绑定在了一起，确保输入形状100%匹配
    model_stacking, expected_cols = joblib.load(stacking_path)
    model_xgboost, _ = joblib.load(xgb_path)
    return model_stacking, model_xgboost, expected_cols

model_stacking, model_xgboost, expected_cols = load_models()

if model_stacking is None:
    st.error("⚠️ Error: Model files not found! Please ensure 'model_stacking.pkl' and 'model_xgboost_severe.pkl' are in the same directory as this app.py.")
    st.stop()

# ==========================================
# 3. 侧边栏：患者特征输入
# ==========================================
st.sidebar.header("🩺 Patient Parameters")

age = st.sidebar.number_input("Age (years)", value=60, min_value=18, max_value=100)
gender = st.sidebar.selectbox("Sex", options=[("Male", 1), ("Female", 0)], format_func=lambda x: x[0])[1]
bmi = st.sidebar.number_input("Body Mass Index (BMI)", value=22.0, min_value=10.0, max_value=50.0)
diabetes = st.sidebar.selectbox("Diabetes Mellitus", options=[("No", 0), ("Yes", 1)], format_func=lambda x: x[0])[1]

preop_alb = st.sidebar.slider("Preoperative Albumin (g/L)", min_value=20.0, max_value=50.0, value=35.0)
op_time = st.sidebar.slider("Operation Time (minutes)", min_value=30, max_value=600, value=180)

nonop_transfusion = st.sidebar.selectbox("Non-operative Blood Transfusion", options=[("No", 0), ("Yes", 1)], format_func=lambda x: x[0])[1]
intraop_plasma = st.sidebar.number_input("Intraoperative Plasma Transfusion (mL)", value=800, min_value=0, max_value=5000)
nonop_alb = st.sidebar.number_input("Non-operative Albumin Infusion (g)", value=20, min_value=0, max_value=100)
vasoactive = st.sidebar.selectbox("Perioperative Vasoactive Agents", options=[("No", 0), ("Yes", 1)], format_func=lambda x: x[0])[1]

tnm_stage = st.sidebar.selectbox("TNM Stage", options=[("I", 1), ("II", 2), ("III", 3), ("IV", 4)], format_func=lambda x: x[0])[1]
surgery_type = st.sidebar.selectbox("Surgery Type", options=[("Gastric", 0), ("Colorectal", 1)], format_func=lambda x: x[0])[1]

# ==========================================
# 4. 主界面：预测逻辑与双模型输出
# ==========================================
if st.sidebar.button("Calculate Dual-Risk", type="primary"):
    with st.spinner("Analyzing patient data through Stacking and XGBoost ensembles..."):
        
        # 将用户输入按顺序打包
        raw_inputs = [age, gender, bmi, diabetes, preop_alb, op_time, 
                      nonop_transfusion, intraop_plasma, nonop_alb, vasoactive, tnm_stage, surgery_type]
        
        # 将数据映射到模型真正需要的列名上
        input_data = pd.DataFrame([raw_inputs], columns=expected_cols)

        # 🚀 真实模型推理！
        risk_overall = model_stacking.predict_proba(input_data)[0][1]
        risk_severe = model_xgboost.predict_proba(input_data)[0][1]
        
        # 逻辑约束：重症的概率在数学上不能大于患病的总体概率
        risk_severe = min(risk_overall, risk_severe) 

        # --- 结果展示区 ---
        col1, col2 = st.columns(2)
        
        # 面板 1：总体 AKI 风险 (Stacking)
        with col1:
            st.markdown("### 🔵 Overall AKI Risk")
            st.caption("Powered by Stacking Ensemble Meta-Learner")
            
            if risk_overall >= 0.20:
                st.error(f"## {risk_overall * 100:.1f}%\n**HIGH RISK**")
            elif risk_overall >= 0.10:
                st.warning(f"## {risk_overall * 100:.1f}%\n**MODERATE RISK**")
            else:
                st.success(f"## {risk_overall * 100:.1f}%\n**LOW RISK**")
                
        # 面板 2：重症 AKI 风险 (XGBoost)
        with col2:
            st.markdown("### 🔴 Severe AKI Risk (Stage II/III)")
            st.caption("Powered by Extreme Gradient Boosting (XGBoost)")
            
            if risk_severe >= 0.10:
                st.error(f"## {risk_severe * 100:.1f}%\n**HIGH SEVERITY ALERT**")
                st.info("💡 **Clinical Action:** ICU monitoring, strict fluid management, and nephrology consultation recommended.")
            else:
                st.success(f"## {risk_severe * 100:.1f}%\n**Low Severity Risk**")

        st.divider()
        st.markdown("### 📋 Patient Profile Summary")
        st.dataframe(input_data, hide_index=True)

2026-05-17 13:50:59.354 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.356 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.358 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.359 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.360 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-17 13:50:59.362 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar